## Olist E-Commerce — RFM Analysis
This notebook loads the Olist Brazilian e-commerce dataset using DuckDB to query relational CSV files and produce a customer-level RFM (Recency, Frequency, Monetary) table for downstream segmentation analysis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Mount Google Drive and verify data

In [ ]:
!pip install duckdb

import duckdb
import pandas as pd

# Verify files are accessible
import os

data_path = '/content/drive/MyDrive/ecommerce-segmentation/data/'

files = os.listdir(data_path)
print(files)

['olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_products_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'product_category_name_translation.csv', 'olist_sellers_dataset.csv']


### Build RFM Table with SQL
We join three tables (orders, customers, payments) and aggregate to one row per unique customer. Recency is measured in days since last purchase, relative to the dataset end date of October 17, 2018.

In [ ]:
con = duckdb.connect()

rfm = con.execute(f"""
    WITH payments AS (
        SELECT
            order_id,
            SUM(payment_value) AS order_value
        FROM read_csv_auto('{data_path}olist_order_payments_dataset.csv')
        GROUP BY order_id
    ),

    orders AS (
        SELECT
            order_id,
            customer_id,
            order_purchase_timestamp
        FROM read_csv_auto('{data_path}olist_orders_dataset.csv')
        WHERE order_status = 'delivered'
    ),

    customers AS (
        SELECT
            customer_id,
            customer_unique_id,
            customer_city,
            customer_state
        FROM read_csv_auto('{data_path}olist_customers_dataset.csv')
    ),

    joined AS (
        SELECT
            c.customer_unique_id,
            c.customer_city,
            c.customer_state,
            o.order_purchase_timestamp,
            p.order_value
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN payments p ON o.order_id = p.order_id
    ),

    rfm AS (
        SELECT
            customer_unique_id,
            customer_city,
            customer_state,
            DATE_DIFF('day', MAX(order_purchase_timestamp::DATE), DATE '2018-10-17') AS recency,
            COUNT(*) AS frequency,
            ROUND(SUM(order_value), 2) AS monetary
        FROM joined
        GROUP BY customer_unique_id, customer_city, customer_state
    )

    SELECT * FROM rfm
    ORDER BY monetary DESC
""").df()

print(rfm.head(10))
print(f"\nTotal customers: {len(rfm)}")

                 customer_unique_id   customer_city customer_state  recency  \
0  0a0a92112bd4c708ca5fde585afaa872  rio de janeiro             RJ      383   
1  da122df9eeddfedc1dc1f5349a1a690c        araruama             RJ      564   
2  763c8b1c9c68a0229c42c9fc6f662b93      vila velha             ES       94   
3  dc4802a71eae9be1dd28f5d788ceb526    campo grande             MS      612   
4  459bef486812aa25204be022145caa62         vitoria             ES       84   
5  ff4159b92c40ebe40454e3e6a7c35ed6         marilia             SP      511   
6  4007669dec559734d6f53e029e360987     divinopolis             MG      327   
7  eebb5dda148d3893cdaf5b5ca3040ccb            maua             SP      547   
8  48e1ac109decbb87765a3eade6854098     joao pessoa             PB      117   
9  c8460e4251689ba205045f3ea17884a1    porto alegre             RS       70   

   frequency  monetary  
0          1  13664.08  
1          2   7571.63  
2          1   7274.88  
3          1   6929.31  
4    

### Result
The query returns 93,470 unique customers. High monetary values with frequency of 1 suggest most customers are one-time buyers — a pattern we'll explore further in segmentation.

In [ ]:
rfm.to_csv(f'{data_path}rfm_output.csv', index=False)
print("Saved successfully")

Saved successfully
